In [ ]:
import nmrglue as ng
import numpy as np
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
import os
import glob
import pandas as pd
from joblib import load
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import traceback
from datetime import datetime
import sys

# --- Output Directory Detection Logic ---
data_dir = '/home/jovyan/work/data'
external_output_dir = '/home/jovyan/work/output'

# Ensure the external output dir exists to check its device ID
os.makedirs(external_output_dir, exist_ok=True)

# Compare Device IDs to detect if /output is a mounted volume (-v)
root_dev = os.stat('/').st_dev
output_dev = os.stat(external_output_dir).st_dev

if root_dev != output_dev:
    target_dir = external_output_dir
else:
    target_dir = data_dir

# Strictly validate write permissions before proceeding
if not os.access(target_dir, os.W_OK):
    print(f"ERROR: The target directory [{target_dir}] is READ-ONLY.")
    print("Process aborted. Please ensure you have write permissions or mount an output volume (-v).")
    raise PermissionError(f"Target directory [{target_dir}] is read-only.")

milog = os.path.join(target_dir, "000ERRORS.log")

def log_error(msg):
    print(f"⚠️ {msg}")
    try:
        with open(milog, "a", encoding="utf-8") as f:
            f.write(msg + "\n")
    except Exception:
        pass

# Console output
terminal_output = open('/dev/stdout', 'w')

METABOLITE_SETTINGS = {
 '1,5-Anhydrosorbitol': {'coeff_a': 151572.089, 'j_range': (10.7, 11.7), 'ppm_range': (3.291, 3.2935)},
 '2-Aminobutyric acid': {'coeff_a': 160686.562, 'j_range': (7.4, 8.3), 'ppm_range': (0.994, 1.000)},
 '2-Hydroxybutyric acid': {'coeff_a': 228285.114, 'j_range': (6.5, 8.5), 'ppm_range': (0.913, 0.918)},
 '2-Oxoglutaric acid': {'coeff_a': 159799.580, 'j_range': (6, 9), 'ppm_range': (3.02, 3.033)},
 '3-Hydroxybutyric acid': {'coeff_a': 817687.794, 'j_range': (3, 3.4), 'ppm_range': (1.21, 1.22)},
 '3-Hydroxyisobutyric acid': {'coeff_a': 767808.839, 'j_range': (3.7, 4), 'ppm_range': (1.084, 1.092)},
 'Acetic acid': {'coeff_a': 1474071.034, 'j_range': (-0.25, 0.25), 'ppm_range': (1.93, 1.937)},
 'Acetoacetic acid': {'coeff_a': 686648.853, 'j_range': (-0.3, 0.3), 'ppm_range': (2.293, 2.302)},
 'Acetone': {'coeff_a': 1847216.874, 'j_range': (-0.25, 0.25), 'ppm_range': (2.24, 2.25)},
 'Alanine': {'coeff_a': 771015.601, 'j_range': (3.75, 3.9), 'ppm_range': (1.49, 1.50)},
 'Arginine': {'coeff_a': 26443.916, 'j_range': (13.8, 15.6), 'ppm_range': (1.92, 1.928)},
 'Asparagine': {'coeff_a': 121982.067, 'j_range': (4, 5.1), 'ppm_range': (2.84, 2.857)},
 'Aspartate': {'coeff_a': 137414.16, 'j_range': (12.9, 13.9), 'ppm_range': (2.68, 2.69)},
 'Choline': {'coeff_a': 5777443.281, 'j_range': (-0.2, 0.2), 'ppm_range': (3.208, 3.215)},
 'Betaine': {'coeff_a': 551917.597, 'j_range': (-0.25, 0.25), 'ppm_range': (3.90, 3.915)},
 'Citric acid': {'coeff_a': 286934.462, 'j_range': (7.7, 8.7), 'ppm_range': (2.69, 2.698)},
 'Creatine': {'coeff_a': 1312083.588, 'j_range': (-0.25, 0.25), 'ppm_range': (3.943, 3.946)},
 'Creatinine': {'coeff_a': 542564.295, 'j_range': (-0.25, 0.25), 'ppm_range': (4.06, 4.08)},
 'Cystine': {'coeff_a': 221635.742, 'j_range': (5.9, 6.8), 'ppm_range': (4.062, 4.070)},
 'D-Galactose': {'coeff_a': 212153.425, 'j_range': (3.6, 4.8), 'ppm_range': (4.60, 4.61)},
 'Dimethylamine': {'coeff_a': 212922.696, 'j_range': (-0.25, 0.25), 'ppm_range': (2.739, 2.746)},
 'Dimethylsulfone': {'coeff_a': 2528743.623, 'j_range': (-0.2, 0.2), 'ppm_range': (3.163, 3.174)},
 'Ethanol': {'coeff_a': 292021.15, 'j_range': (6.65, 7.85), 'ppm_range': (1.195, 1.203)},
 'Formic acid': {'coeff_a': 351246.19, 'j_range': (-0.25, 0.25), 'ppm_range': (8.47, 8.48)},
 'Glucose': {'coeff_a': 154619.813, 'j_range': (5.8, 6.6), 'ppm_range': (3.917, 3.92)},
 'Glutamic acid': {'coeff_a': 122884.449, 'j_range': (7.4, 9), 'ppm_range': (2.37, 2.377)},
 'Glutamine': {'coeff_a': 50588.016, 'j_range': (10.2, 11.3), 'ppm_range': (2.469, 2.476)},
 'Glycerol': {'coeff_a': 112535.018, 'j_range': (8.5, 10.2), 'ppm_range': (3.58, 3.589)},
 'Glycine': {'coeff_a': 971674.457, 'j_range': (-0.1, 0.2), 'ppm_range': (3.575, 3.582)},
 'Histidine': {'coeff_a': 70443.485, 'j_range': (10.8, 13), 'ppm_range': (3.128, 3.138)},
 'Isoleucine': {'coeff_a': 984393.578, 'j_range': (3.7, 3.9), 'ppm_range': (1.023, 1.026)},
 'Lactic acid': {'coeff_a': 942764.55, 'j_range': (3.7, 3.9), 'ppm_range': (1.343, 1.346)},
 'Leucine': {'coeff_a': 498419.788, 'j_range': (3.1, 3.9), 'ppm_range': (0.98, 0.985)},
 'Lysine': {'coeff_a': 110570.815, 'j_range': (7.3, 8.3), 'ppm_range': (3.043, 3.052)},
 'Methanol': {'coeff_a': 1443543.75, 'j_range': (-0.25, 0.25), 'ppm_range': (3.375, 3.382)},
 'Methionine': {'coeff_a': 495151.918, 'j_range': (-0.25, 0.25), 'ppm_range': (2.155, 2.1585)},
 'Myo-inositol': {'coeff_a': 156618.83, 'j_range': (9, 10), 'ppm_range': (3.30, 3.31)},
 'N,N-Dimethylglycine': {'coeff_a': 2468334.986, 'j_range': (-0.25, 0.25), 'ppm_range': (2.939, 2.946)},
 'Ornithine': {'coeff_a': 164003.059, 'j_range': (7.3, 8.3), 'ppm_range': (3.0725, 3.08)},
 'Phenylalanine': {'coeff_a': 69594.012, 'j_range': (7.4, 8.4), 'ppm_range': (7.438, 7.45)},
 'Proline': {'coeff_a': 56864.7, 'j_range': (7.4, 8.4), 'ppm_range': (4.143, 4.152)},
 'Pyruvic acid': {'coeff_a': 975617.63, 'j_range': (-0.25, 0.25), 'ppm_range': (2.385, 2.39)},
 'Sarcosine_CORR': {'coeff_a': 1510318.936, 'j_range': (5.4, 7.8), 'ppm_range': (2.75, 2.77)},
 'Sarcosine': {'coeff_a': 1812382.723, 'j_range': (-0.25, 0.25), 'ppm_range': (2.75, 2.77)},
 'Serine': {'coeff_a': 116686.4493, 'j_range': (8.8, 9.5), 'ppm_range': (3.957, 3.963)},
 'Succinic acid': {'coeff_a': 2616720.995, 'j_range': (-0.2, 0.2), 'ppm_range': (2.415, 2.423)},
 'Threonine': {'coeff_a': 61820.889, 'j_range': (5.5, 6.6), 'ppm_range': (4.261, 4.269)},
 'Trimethylamine-N-oxide_CORR': {'coeff_a': 551917.597, 'j_range': (-0.25, 0.25), 'ppm_range': (3.90, 3.915)},
 'Trimethylamine-N-oxide': {'coeff_a': 3949277.029, 'j_range': (-0.25, 0.25), 'ppm_range': (3.275, 3.282)},
 'Tyrosine': {'coeff_a': 240590.617, 'j_range': (4.1, 5.1), 'ppm_range': (7.208, 7.215)},
 'Valine': {'coeff_a': 961607.554, 'j_range': (3.75, 3.9), 'ppm_range': (1.055, 1.06)}
}

for values in METABOLITE_SETTINGS.values():
    if 'coeff_b' not in values:
        values['coeff_b'] = 0
    if 'ext_j_range' not in values:
        j_min, j_max = values['j_range']
        j_center = (j_min + j_max) / 2
        values['ext_j_range'] = [round(j_center - 6, 1), round(j_center + 6, 1)]

BIOMARKER_SETTINGS = {
    'Total cholesterol': {'model': '/home/jovyan/work/modelChol.bin', 'error': 17.319, 'units': 'mg/dl'},
    'HDL cholesterol': {'model': '/home/jovyan/work/modelHDL.bin', 'error': 6.661, 'units': 'mg/dl'},
    'LDL cholesterol': {'model': '/home/jovyan/work/modelLDL.bin', 'error': 14.272, 'units': 'mg/dl'},
    'Triglycerides': {'model': '/home/jovyan/work/modelTG.bin', 'error': 36.932, 'units': 'mg/dl'},
    'Total protein': {'model': '/home/jovyan/work/modelProt.bin', 'error': 0.4953, 'units': 'g/dl'},
    'Albumin': {'model': '/home/jovyan/work/modelAlbumin.bin', 'error': 0.2439, 'units': 'g/dl'},
    'Hemoglobin': {'model': '/home/jovyan/work/modelHemo.bin', 'error': 1.260, 'units': 'g/dl'},
    'Urea': {'model': '/home/jovyan/work/modelUrea.bin', 'error': 7.667, 'units': 'mg/dl'},
    'Urate': {'model': '/home/jovyan/work/modelUrate.bin', 'error': 1.329, 'units': 'mg/dl'},
    'Calcium': {'model': '/home/jovyan/work/modelCa.bin', 'error': 0.5728, 'units': 'mg/dl'},
    'C reactive protein': {'model': '/home/jovyan/work/modelCrp.bin', 'error': 1.298, 'units': 'mg/dl'},
    'Glyc A': {'model': '/home/jovyan/work/modelGlycA.bin', 'error': 0.01612, 'units': 'mM'},
    'Glyc B': {'model': '/home/jovyan/work/modelGlycB.bin', 'error': 0.00676, 'units': 'mM'},
    'SPC': {'model': '/home/jovyan/work/modelSPC.bin', 'error': 0.01547, 'units': 'mM'},
    'Transferrin': {'model': '/home/jovyan/work/modelTransfe.bin', 'error': 45.036, 'units': 'mg/dl'},
    'Iron': {'model': '/home/jovyan/work/modelFe.bin', 'error': 34.085, 'units': 'µg/dl'},
    'Fructosamine': {'model': '/home/jovyan/work/modelFruc.bin', 'error': 38.490, 'units': 'µM'},
    'Apolipoprotein B': {'model': '/home/jovyan/work/modelApoB.bin', 'error': 0.2906, 'units': 'mg/ml'},
    'Lipoprotein(a)': {'model': '/home/jovyan/work/modelLPA.bin', 'error': 21.693, 'units': 'mg/dl'},
    'Erythrocyte sedimentation rate': {'model': '/home/jovyan/work/modelEsr.bin', 'error': 6.922, 'units': 'mm/h'},
    'Bilirubin': {'model': '/home/jovyan/work/modelBili.bin', 'error': 0.4003, 'units': 'mg/dl'},
    'Estimated Glomerular Filtration Rate': {'model': '/home/jovyan/work/modelEgfr.bin', 'error': 16.255, 'units': 'ml/min/1.73m²'},
    'Leukocytes': {'model': '/home/jovyan/work/modelLeuco.bin', 'error': 2.227, 'units': 'G/l'},
    'Erythrocytes': {'model': '/home/jovyan/work/modelEritro.bin', 'error': 0.485, 'units': 'T/l'},
    'Platelets': {'model': '/home/jovyan/work/modelPlat.bin', 'error': 68.714, 'units': 'G/l'}
}

def readspectrum(path, num):
    dic, data = ng.bruker.read_pdata(path+'/'+str(num)+'/pdata/1')
    start_ppm = float(dic['procs']['OFFSET'])
    end_ppm = start_ppm - float(dic['acqus']['SW'])
    ftsize_ppm = dic['procs']['FTSIZE']
    step_ppm = float(dic['acqus']['SW']) / ftsize_ppm
    ppm_values = np.arange(start_ppm, end_ppm, -step_ppm)[:ftsize_ppm]
    
    start_j = float(dic['proc2s']['OFFSET'])
    end_j = start_j - float(dic['acqu2s']['SW'])
    ftsize_j = dic['proc2s']['FTSIZE']
    step_j = float(dic['acqu2s']['SW']) / ftsize_j
    j_values = np.arange(start_j, end_j, -step_j)[:ftsize_j]

    return dic, data, ppm_values, j_values

# Define 1D Lorentzian function with offset
def lorentzian_1d_with_offset(x, x0, gamma, A, offset):
    return A * (gamma**2 / ((x - x0)**2 + gamma**2)) + offset

def read1D(spectrum_path, ppm_reference_path):
    dic, data = ng.bruker.read_pdata(spectrum_path)
    start = float(dic['procs']['OFFSET'])
    end = start - float(dic['acqus']['SW'])
    ftsize = dic['procs']['FTSIZE']
    step = float(dic['acqus']['SW']) / ftsize
    ppms = np.arange(start, end, -step)[:ftsize]

    ppm_range = (15.1, 14.9)
    ppm_mask = (ppms >= ppm_range[1]) & (ppms <= ppm_range[0])
    data_masked = data[:len(ppms)]
    try:
        popt_1d, pcov_1d = curve_fit(lorentzian_1d_with_offset, ppms[ppm_mask], data_masked[ppm_mask], p0=[15, 0.001, np.max(data_masked[ppm_mask]), np.min(data_masked[ppm_mask])])
    except RuntimeError:
        print("1D Lorentzian fit could not be performed on the 1D spectrum")
        popt_1d, pcov_1d = [None]*4, None

    if popt_1d[0] is not None:
        x0_1d, gamma_1d, A_1d, offset_1d = popt_1d
        data_corrected = data_masked[ppm_mask] - offset_1d
        ppm_values_corrected = ppms[ppm_mask]
        eretic_factor = np.sum(lorentzian_1d_with_offset(ppm_values_corrected, x0_1d, gamma_1d, A_1d, 0)) * np.abs(ppm_values_corrected[0] - ppm_values_corrected[1])

    ppm_ref = pd.read_csv(ppm_reference_path, header=None).to_numpy().squeeze()
    data_interp = np.flip(np.interp(np.flip(ppm_ref), np.flip(ppms), np.flip(data)))

    normINT = data_interp[20570:20572].max() / 10000
    data_interp /= normINT

    ppms_cal, data_cal = calibrate_spectrum(ppm_ref, data_interp)
    data_cal = np.round_(data_cal, decimals=5)

    return ppms_cal, data_cal, eretic_factor

# Define baseline correction function
def baseline_asls(y, lam=1e4, p=0.01, n_iter=10):
    L = len(y)
    D = np.zeros((L, L))
    for i in range(L - 2):
        D[i, i] = 1
        D[i, i + 1] = -2
        D[i, i + 2] = 1
    D = lam * D.T @ D
    w = np.ones(L)
    for _ in range(n_iter):
        W = np.diag(w)
        Z = np.linalg.solve(W + D, w * y)
        w = p * (y > Z) + (1 - p) * (y < Z)
    return Z

def calibrate_spectrum(ppms, spectrum, alanine_range=(1.48, 1.52), target_ppm=1.496, ppm_range=(10.95, -0.95)):
    alanine_mask = (ppms >= alanine_range[0]) & (ppms <= alanine_range[1])
    if not np.any(alanine_mask):
        return ppms, spectrum  
    
    data_alanine = spectrum[alanine_mask]
    baseline = baseline_asls(data_alanine, lam=1e5, p=0.01, n_iter=10)
    data_alanine_corrected = data_alanine - baseline
    
    peak_indices, _ = find_peaks(data_alanine_corrected, height=np.max(data_alanine_corrected) * 0.5)
    
    if len(peak_indices) == 2:
        alanine_peaks = ppms[alanine_mask][peak_indices]
        alanine_center = np.mean(alanine_peaks)
        calibration_shift = target_ppm - alanine_center

        shifted_ppms = ppms + calibration_shift
        interpolator = interp1d(shifted_ppms, spectrum, bounds_error=False, fill_value=0)
        calibrated_spectrum = interpolator(ppms)
    else:
        calibrated_spectrum = spectrum  

    ppm_mask = (ppms >= ppm_range[1]) & (ppms <= ppm_range[0])
    ppms_cut = ppms[ppm_mask]
    spectrum_cut = calibrated_spectrum[ppm_mask]

    return ppms_cut, spectrum_cut

def bin1D(ppm_array, intensity_array, bin_size=100):
    ppm_padded = np.pad(ppm_array.astype(float), 
                        (0, (bin_size - len(ppm_array) % bin_size) % bin_size), 
                        mode='constant', constant_values=np.nan)
    intensity_padded = np.pad(intensity_array.astype(float), 
                              (0, (bin_size - len(intensity_array) % bin_size) % bin_size), 
                              mode='constant', constant_values=np.nan)

    ppm_binned = np.nanmean(ppm_padded.reshape(-1, bin_size), axis=1)
    intensity_binned = np.nanmean(intensity_padded.reshape(-1, bin_size), axis=1)

    ppm_binned = np.round(ppm_binned, decimals=5)
    intensity_binned = np.round(intensity_binned, decimals=5)

    return ppm_binned, intensity_binned


def getintegral(sample, metname, data, ppm_values, j_values, settings):
    ppm_range = settings['ppm_range']
    j_range = settings['j_range']
    ext_j_range = settings['ext_j_range']
    a1 = settings['coeff_a']
    b1 = settings['coeff_b']
    is_wider = settings.get('wider', False)
    
    def gaussian_1d(x, mu, sigma, amplitude):
        return amplitude * np.exp(-(x - mu)**2 / (2 * sigma**2))
    def super_gaussian_1d(x, mu, sigma, amplitude, p):
        return amplitude * np.exp(-(np.abs(x - mu)**p) / (2 * sigma**p))
        
    start_ppm_metabolite = ppm_range[0]
    end_ppm_metabolite = ppm_range[1]
    start_j_metabolite = j_range[0]
    end_j_metabolite = j_range[1]

    ppm_indices_metabolite = np.where((ppm_values >= start_ppm_metabolite) & (ppm_values <= end_ppm_metabolite))[0]
    j_indices_metabolite = np.where((j_values >= start_j_metabolite) & (j_values <= end_j_metabolite))[0]
    
    sub_data_metabolite = data[np.ix_(j_indices_metabolite, ppm_indices_metabolite)]
    sub_ppm_metabolite = ppm_values[ppm_indices_metabolite]
    sub_j_metabolite = j_values[j_indices_metabolite]
    
    peak_index_metabolite = np.unravel_index(np.argmax(sub_data_metabolite, axis=None), sub_data_metabolite.shape)
    peak_ppm_metabolite = sub_ppm_metabolite[peak_index_metabolite[1]]
    peak_j_metabolite = sub_j_metabolite[peak_index_metabolite[0]]
    
    start_j_metabolite_ext = ext_j_range[0]
    end_j_metabolite_ext = ext_j_range[1]
    
    j_indices_metabolite_ext = np.where((j_values >= start_j_metabolite_ext) & (j_values <= end_j_metabolite_ext))[0]
    section_data_metabolite_ext = data[j_indices_metabolite_ext, ppm_indices_metabolite[peak_index_metabolite[1]]]
    
    fit_region_indices = (j_values[j_indices_metabolite_ext] >= start_j_metabolite_ext) & (j_values[j_indices_metabolite_ext] <= end_j_metabolite_ext)
    fit_j_values = j_values[j_indices_metabolite_ext][fit_region_indices]
    fit_section_data = section_data_metabolite_ext[fit_region_indices]
    
    min_value = np.min(fit_section_data)
    fit_section_data_normalized = fit_section_data - min_value

    delta_x = 3  
    mask = (fit_j_values >= peak_j_metabolite - delta_x) & (fit_j_values <= peak_j_metabolite + delta_x)
    fit_j_values_restricted = fit_j_values[mask]
    fit_section_data_restricted = fit_section_data_normalized[mask]
    
    try:
        delta_x0 = 0.9  
        
        if is_wider:
            p0 = [peak_j_metabolite, 0.6, np.max(fit_section_data_restricted), 3.0]
            popt_metabolite, pcov_metabolite = curve_fit(
                super_gaussian_1d, fit_j_values_restricted, fit_section_data_restricted, 
                p0=p0, bounds=([peak_j_metabolite - delta_x0, 0, -np.inf, 2.0], [peak_j_metabolite + delta_x0, 2.5, np.inf, 6.0])
            )
        else:
            p0 = [peak_j_metabolite, 0.6, np.max(fit_section_data_restricted)]
            popt_metabolite, pcov_metabolite = curve_fit(
                gaussian_1d, fit_j_values_restricted, fit_section_data_restricted, 
                p0=p0, bounds=([peak_j_metabolite - delta_x0, 0, -np.inf], [peak_j_metabolite + delta_x0, 1.2, np.inf])
            )
        fitOK = True
    except RuntimeError:
        try:
            delta_x = 2
            mask = (fit_j_values >= peak_j_metabolite - delta_x) & (fit_j_values <= peak_j_metabolite + delta_x)
            fit_j_values_restricted = fit_j_values[mask]
            fit_section_data_restricted = fit_section_data_normalized[mask]
            delta_x0 = 0.9
            
            if is_wider:
                p0 = [peak_j_metabolite, 0.6, np.max(fit_section_data_restricted), 3.0]
                popt_metabolite, pcov_metabolite = curve_fit(
                    super_gaussian_1d, fit_j_values_restricted, fit_section_data_restricted, 
                    p0=p0, bounds=([peak_j_metabolite - delta_x0, 0, -np.inf, 2.0], [peak_j_metabolite + delta_x0, 3.0, np.inf, 6.0])
                )
            else:
                p0 = [peak_j_metabolite, 0.6, np.max(fit_section_data_restricted)]
                popt_metabolite, pcov_metabolite = curve_fit(
                    gaussian_1d, fit_j_values_restricted, fit_section_data_restricted, 
                    p0=p0, bounds=([peak_j_metabolite - delta_x0, 0, -np.inf], [peak_j_metabolite + delta_x0, 1.5, np.inf])
                )
            fitOK = True
        except RuntimeError:
            fitOK = False
            if is_wider:
                popt_metabolite, pcov_metabolite = [peak_j_metabolite, 1000000, 0, 2], np.eye(4)*1000000
            else:
                popt_metabolite, pcov_metabolite = [peak_j_metabolite, 1000000, 0], [[1000000, 0, 0],[0, 1000000, 0],[0, 0, 1000000]]

    if is_wider:
        x0_metabolite, sigma_metabolite, A_metabolite, p_metabolite = popt_metabolite
    else:
        x0_metabolite, sigma_metabolite, A_metabolite = popt_metabolite

    j_values_metabolite_ext = np.linspace(start_j_metabolite_ext, end_j_metabolite_ext, 500)
    
    if is_wider:
        ZZ_metabolite_corr = super_gaussian_1d(j_values_metabolite_ext, x0_metabolite, sigma_metabolite, A_metabolite, p_metabolite)
    else:
        ZZ_metabolite_corr = gaussian_1d(j_values_metabolite_ext, x0_metabolite, sigma_metabolite, A_metabolite)
        
    integral_metabolite_corr = np.sum(ZZ_metabolite_corr) * (j_values_metabolite_ext[1] - j_values_metabolite_ext[0])
    
    half_width = 0.5 * (end_j_metabolite_ext - start_j_metabolite_ext)
    central_start = x0_metabolite - half_width / 2
    central_end = x0_metabolite + half_width / 2
    
    central_mask = (fit_j_values >= central_start) & (fit_j_values <= central_end)
    central_j_values = fit_j_values[central_mask]
    central_fit_data = fit_section_data_normalized[central_mask]
    
    if is_wider:
        central_predictions = super_gaussian_1d(central_j_values, x0_metabolite, sigma_metabolite, A_metabolite, p_metabolite)
    else:
        central_predictions = gaussian_1d(central_j_values, x0_metabolite, sigma_metabolite, A_metabolite)
    
    central_residuales = central_fit_data - central_predictions
    std_residuales = np.std(central_residuales)
    error_integral_metabolite = abs(std_residuales * (central_j_values[-1] - central_j_values[0]))
    
    if not fitOK:
        integral_metabolite_corr = 0.000001
        error_integral_metabolite = 1000000

    met = (integral_metabolite_corr - b1) / a1
    error_met = error_integral_metabolite / a1

    if met < 0:
        met = 0
    if not fitOK:
        met=0.001
        error_met=10
        
    return met, error_met, peak_ppm_metabolite, peak_j_metabolite

def predict_with_model(model_path, ppm_values, data, scaler_path=None):
    data = data.reshape(1, -1)
    model = load(model_path)
    
    if scaler_path:
        scaler = load(scaler_path)
        data_transformed = scaler.transform(data)
    else:
        data_transformed = data

    prediction = model.predict(data_transformed)
    return prediction[0]

def clip_negative(y):
    return np.clip(y, 0, None)

def identity(y):
    return y
    
# Main paths
now = datetime.now()
nowd = now.strftime("%Y%m%d")

subdirectories = sorted([os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])

results = []

filtered_metabolite_settings = {key: value for key, value in METABOLITE_SETTINGS.items() if not key.endswith('_CORR')}

any_success = False

for count, subdirectory in enumerate(subdirectories, start=1):
    status = "ok"
    err_msg = ""
    try:
        noesy_expn, jres_expn = '10', '12'
        for item in sorted(os.listdir(subdirectory)):
            if item.isdigit() and os.path.isdir(os.path.join(subdirectory, item)):
                acqus_path = os.path.join(subdirectory, item, "acqus")
                if os.path.exists(acqus_path):
                    with open(acqus_path, 'r', errors='ignore') as file:
                        content = file.read()
                        if "jresgpprqf" in content:
                            jres_expn = item
                            break
                            
        if not (os.path.exists(os.path.join(subdirectory, jres_expn))):
            continue
    
        dic, data, ppm_values, j_values = readspectrum(data_dir+'/'+os.path.basename(subdirectory), jres_expn)
        magnet_field = dic.get("acqus", {}).get("BF1", None)
        j_values = j_values * magnet_field
        
        sample=os.path.basename(subdirectory)
    
        # NOESY processing
        for item in sorted(os.listdir(subdirectory)):
            if item.isdigit() and os.path.isdir(os.path.join(subdirectory, item)):
                acqus_path = os.path.join(subdirectory, item, "acqus")
                if os.path.exists(acqus_path):
                    with open(acqus_path, 'r', errors='ignore') as file:
                        content = file.read()
                        if "noesygppr1d" in content:
                            noesy_expn = item
                            break
                            
        if not (os.path.exists(os.path.join(subdirectory, noesy_expn))):
            continue
    
        ppmsN, dataN, eretic_factor= read1D(subdirectory+'/'+noesy_expn+'/pdata/1', '/home/jovyan/work/dataPPMS_NOESY.csv')
        data = data*40291.326/eretic_factor 
    
        alanine_ppm_range = (1.45, 1.55) 
        alanine_j_range = (1.8, 6.0)     
        
        ppm_mask = (ppm_values >= alanine_ppm_range[0]) & (ppm_values <= alanine_ppm_range[1])
        j_mask = (j_values >= alanine_j_range[0]) & (j_values <= alanine_j_range[1])
        
        ppm_values_filtered = ppm_values[ppm_mask]
        data_region = data[j_mask, :][:, ppm_mask]
        
        max_index_j, max_index_ppm = np.unravel_index(np.argmax(data_region), data_region.shape)
        max_ppm = ppm_values_filtered[max_index_ppm]
        calibration_shift = 1.496 - max_ppm
        ppm_values = ppm_values + calibration_shift
    
        metabolite_results = {}
        metabolite_corrections = {}
    
        for metabolite, settings in METABOLITE_SETTINGS.items():
            concentration, error, chemical_shift, J = getintegral(sample, metabolite, data, ppm_values, j_values, settings)
            
            metabolite_results[metabolite] = {
                "concentration": concentration,
                "error": error,
                "chemical_shift": chemical_shift,
                "J": J
            }
        
            if metabolite.endswith('_CORR'):
                metabolite_corrections[metabolite] = metabolite_results.pop(metabolite)
                
        for corrected_metabolite, correction_data in metabolite_corrections.items():
            metabolite_name = corrected_metabolite.replace('_CORR', '')
            if metabolite_name in metabolite_results:
                metabolite_results[metabolite_name]["error"] += correction_data["error"]
                if metabolite_results[metabolite_name]["concentration"] > correction_data["concentration"]:
                    metabolite_results[metabolite_name]["concentration"] -= correction_data["concentration"]
                else:
                    metabolite_results[metabolite_name]["concentration"] = 0.0001
        
        model_values = {}
        ppm_values2, data2 = bin1D(ppmsN, dataN, bin_size=100)
        
        for metabolite, settings in BIOMARKER_SETTINGS.items():
            model=settings.get("model")
            error=settings.get("error")
            if metabolite == 'Lipoprotein(a)':
                ppmsLPA = np.loadtxt('/home/jovyan/work/ppmsLPA.txt')
                dataLPA = np.array([dataN[ppmsN == ppm][0] for ppm in ppmsLPA])
                prediction = predict_with_model(model, ppmsLPA, dataLPA)
            else:
                prediction = predict_with_model(model, ppm_values2, data2)
            unit=settings.get("units")
    
            model_values[metabolite] = {"value": f"{prediction:.5f}", "error": f"{error:.5e}", "units": unit}
    
        fid_path = os.path.join(subdirectory, jres_expn, 'ser')
        if os.path.exists(fid_path):
            modification_time = os.path.getmtime(fid_path)
            modification_datep = datetime.fromtimestamp(modification_time).date() 
            modification_date = modification_datep.strftime("%d/%m/%Y")
        else:
            modification_date = None
        
        results_row = [os.path.basename(subdirectory), modification_date]
        
        for metabolite in filtered_metabolite_settings.keys():
            concentration = metabolite_results[metabolite]["concentration"]
            error = metabolite_results[metabolite]["error"]
            results_row.append(concentration)
            results_row.append(error)
    
        # Negative limits for clinical parameters
        if float(model_values["Total cholesterol"]["value"]) < 0:
            model_values["Total cholesterol"]["value"] = "0.0001" 
            model_values["Total cholesterol"]["error"] = "1000"
        if float(model_values["HDL cholesterol"]["value"]) < 0:
            model_values["HDL cholesterol"]["value"] = "0.0001" 
            model_values["HDL cholesterol"]["error"] = "1000"
        if float(model_values["LDL cholesterol"]["value"]) < 0:
            model_values["LDL cholesterol"]["value"] = "0.0001" 
            model_values["LDL cholesterol"]["error"] = "1000"
        if float(model_values["Triglycerides"]["value"]) < 0:
            model_values["Triglycerides"]["value"] = "0.0001" 
            model_values["Triglycerides"]["error"] = "1000"
        if float(model_values["Hemoglobin"]["value"]) < 0:
            model_values["Hemoglobin"]["value"] = "0.0001" 
            model_values["Hemoglobin"]["error"] = "100000"
        if float(model_values["Total protein"]["value"]) < 0:
            model_values["Total protein"]["value"] = "0.0001" 
            model_values["Total protein"]["error"] = "100000"
        if float(model_values["Albumin"]["value"]) < 0:
            model_values["Albumin"]["value"] = "0.0001" 
            model_values["Albumin"]["error"] = "100000"
        if float(model_values["Urea"]["value"]) < 0:
            model_values["Urea"]["value"] = "0.0001" 
            model_values["Urea"]["error"] = "1000"
        if float(model_values["Urate"]["value"]) < 0:
            model_values["Urate"]["value"] = "0.0001" 
            model_values["Urate"]["error"] = "1000"
        if float(model_values["Calcium"]["value"]) < 0:
            model_values["Calcium"]["value"] = "0.0001" 
            model_values["Calcium"]["error"] = "100"
        if float(model_values["C reactive protein"]["value"]) < 0:
            model_values["C reactive protein"]["value"] = "0.0001" 
            model_values["C reactive protein"]["error"] = "100"
        if float(model_values["Transferrin"]["value"]) < 0:
            model_values["Transferrin"]["value"] = "0.0001" 
            model_values["Transferrin"]["error"] = "1000"
        if float(model_values["Iron"]["value"]) < 0:
            model_values["Iron"]["value"] = "0.0001" 
            model_values["Iron"]["error"] = "1"
        if float(model_values["Fructosamine"]["value"]) < 0:
            model_values["Fructosamine"]["value"] = "0.0001" 
            model_values["Fructosamine"]["error"] = "100"
        if float(model_values["Apolipoprotein B"]["value"]) < 0:
            model_values["Apolipoprotein B"]["value"] = "0.0001" 
            model_values["Apolipoprotein B"]["error"] = "1000"
        if float(model_values["Lipoprotein(a)"]["value"]) < 0:
            model_values["Lipoprotein(a)"]["value"] = "0.0001" 
            model_values["Lipoprotein(a)"]["error"] = "1000"
        if float(model_values["Erythrocyte sedimentation rate"]["value"]) < 0:
            model_values["Erythrocyte sedimentation rate"]["value"] = "0.0001" 
            model_values["Erythrocyte sedimentation rate"]["error"] = "1000"
        if float(model_values["Bilirubin"]["value"]) < 0:
            model_values["Bilirubin"]["value"] = "0.0001" 
            model_values["Bilirubin"]["error"] = "10"
        if float(model_values["Estimated Glomerular Filtration Rate"]["value"]) < 0:
            model_values["Estimated Glomerular Filtration Rate"]["value"] = "0.0001" 
            model_values["Estimated Glomerular Filtration Rate"]["error"] = "1000"
        if float(model_values["Leukocytes"]["value"]) < 0:
            model_values["Leukocytes"]["value"] = "0.0001" 
            model_values["Leukocytes"]["error"] = "100"
        if float(model_values["Erythrocytes"]["value"]) < 0:
            model_values["Erythrocytes"]["value"] = "0.0001" 
            model_values["Erythrocytes"]["error"] = "100"
        if float(model_values["Platelets"]["value"]) < 0:
            model_values["Platelets"]["value"] = "0.0001" 
            model_values["Platelets"]["error"] = "1000"
        if float(model_values["Glyc A"]["value"]) < 0:
            model_values["Glyc A"]["value"] = "0.0001" 
            model_values["Glyc A"]["error"] = "10"
        if float(model_values["Glyc B"]["value"]) < 0:
            model_values["Glyc B"]["value"] = "0.0001" 
            model_values["Glyc B"]["error"] = "10"
        if float(model_values["SPC"]["value"]) < 0:
            model_values["SPC"]["value"] = "0.0001" 
            model_values["SPC"]["error"] = "10"
    
        if float(model_values["Total protein"]["value"]) < float(model_values["Albumin"]["value"]):
            model_values["Total protein"]["value"] = float(model_values["Albumin"]["value"]) 
            model_values["Total protein"]["error"] = 2*float(model_values["Albumin"]["value"])
    
        hdl_value = float(model_values["HDL cholesterol"]["value"])
        ldl_value = float(model_values["LDL cholesterol"]["value"])
        total_cholesterol_value = float(model_values["Total cholesterol"]["value"])
    
        if total_cholesterol_value < (hdl_value + ldl_value):
            model_values["Total cholesterol"]["value"] = f"{hdl_value + ldl_value:.3f}"
            model_values["Total cholesterol"]["error"] = f"{hdl_value + ldl_value:.3f}"
    
        for metabolite, values in model_values.items():
            concentration = values["value"]
            error = values["error"]
            results_row.append(concentration)
            results_row.append(error)
        
        results.append(results_row)
            
        any_success = True
    except Exception as e:
        status = "error"
        err_msg = f"{type(e).__name__}: {e}"
        sample_name = os.path.basename(subdirectory)
        log_error(f"Sample '{sample_name}' skipped due to error: {err_msg}")

        try:
            log_error("Details:\n" + traceback.format_exc(limit=3))
        except Exception:
            pass

        continue

    finally:
        sample_name = os.path.basename(subdirectory)
        if status == "ok":
            print(f"✅ Sample {count}/{len(subdirectories)} completed: {sample_name}", file=terminal_output)
        else:
            print(f"⚠️ Sample {count}/{len(subdirectories)} failed: {sample_name} ({err_msg})", file=terminal_output)
        terminal_output.flush()

if not any_success:
    print("⚠️ No valid samples processed; CSV/Excel generation skipped.")
else:
    columns = ['sample', 'analysis_date']
    
    # Nombres de columnas para los metabolitos
    for metabolite in filtered_metabolite_settings.keys():
        columns.append(f'{metabolite} (mM)')
        columns.append(f'Error {metabolite} (mM)')
    
    # Nombres de columnas para los biomarcadores clínicos
    for metabolite in model_values.keys():
        columns.append(f'{metabolite} ({model_values[metabolite]["units"]})')
        columns.append(f'Error {metabolite} ({model_values[metabolite]["units"]})')
    
    df = pd.DataFrame(results, columns=columns)
    
    # Eliminar la columna de fecha de análisis justo antes de exportar
    df.drop(columns=['analysis_date'], inplace=True)
    
    csv_path = os.path.join(target_dir, 'NMRquantResults.csv')
    df.to_csv(csv_path, index=False)
    
    excel_path = os.path.join(target_dir, 'NMRquantResults.xlsx')
    df.to_excel(excel_path, index=False)
    
    print(f'\nResults successfully saved to:\n - {csv_path}\n - {excel_path}')